In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [1]:
# Load train and test
train_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\train_encoded3.csv")
test_df = pd.read_csv("C:\\Users\\VP678WV\\OneDrive - EY\\Documents\\Delivery_Delay\\data\\processed\\test_encoded3.csv")

NameError: name 'pd' is not defined

In [10]:
train_df.describe()

,profit_per_order,order_item_discount,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_profit_per_order,shipping_mode,distance_normalized,order_to_shipment_days,...,shipment_delay_days,performance_score_order_full_location,performance_score_customer_full_location,performance_score_order_dayofweek,performance_score_shipping_dayofweek,performance_score_order_hour,performance_score_shipping_hour,performance_score_order_daynight,performance_score_ship_daynight,target
count,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,...,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000,16033.000000
mean,0.708227,0.292356,0.303288,0.634309,0.279979,0.365438,0.735297,1.512505,0.389845,0.581467,...,0.433244,0.732309,0.748838,0.488184,0.471761,0.467759,0.483482,0.444088,0.448765,0.085885
std,0.152495,0.267370,0.286959,0.298505,0.360960,0.238130,0.143094,0.865368,0.220542,0.271133,...,0.203595,0.167760,0.068938,0.287778,0.341683,0.207495,0.215749,0.301127,0.302848,0.838013
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000
25%,0.676766,0.082319,0.096401,0.500000,0.000000,0.199996,0.708546,1.000000,0.196349,0.333333,...,0.200000,0.678571,0.743491,0.175435,0.117431,0.368909,0.368322,0.200950,0.247021,-1.000000
50%,0.722225,0.208379,0.120486,0.726190,0.000000,0.368305,0.748808,1.000000,0.373744,0.500000,...,0.400000,0.750000,0.743669,0.573129,0.527232,0.448384,0.486772,0.429931,0.406780,0.000000
75%,0.781034,0.444823,0.457848,0.834812,0.500000,0.578912,0.802872,2.000000,0.524249,0.833333,...,0.600000,0.821429,0.772727,0.611188,0.790442,0.612812,0.596282,0.582074,0.651174,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [11]:
train_df.columns

Index(['profit_per_order', 'order_item_discount', 'order_item_product_price',
       'order_item_profit_ratio', 'order_item_quantity', 'sales',
       'order_profit_per_order', 'shipping_mode', 'distance_normalized',
       'order_to_shipment_days', 'order_shipping_time',
       'order_to_shipment_planned_days', 'shipment_delay_days',
       'performance_score_order_full_location',
       'performance_score_customer_full_location',
       'performance_score_order_dayofweek',
       'performance_score_shipping_dayofweek', 'performance_score_order_hour',
       'performance_score_shipping_hour', 'performance_score_order_daynight',
       'performance_score_ship_daynight', 'payment_type_CASH',
       'payment_type_DEBIT', 'payment_type_PAYMENT', 'payment_type_TRANSFER',
       'target'],
      dtype='object')

In [5]:
# Replace 'target' with your actual target column name if different
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

X_test = test_df.drop('target', axis=1)
y_test = test_df['target']

y_train = y_train.astype('category')
y_test = y_test.astype('category')

In [6]:
# Train a basic RF for feature importance
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Rank features
feature_importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = feature_importances.sort_values(ascending=False)

# Select top N features (e.g., top 15)
selected_features = top_features.head(25).index.tolist()

In [ ]:
X_train_reduced = X_train[selected_features]
X_test_reduced = X_test[selected_features]

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

def objective(trial):
    # Hyperparameter search space
    n_estimators = trial.suggest_categorical('n_estimators', [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000])
    max_depth = trial.suggest_int('max_depth', 3, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])

    # Model
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=42,
        n_jobs=-1
    )
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Cross-validation
    
    score = cross_val_score(
        rf,
        X_train_reduced,
        y_train,
        cv=cv,
        scoring='f1_macro',  # or 'f1_weighted'
        n_jobs=-1
    )

    return score.mean()

In [ ]:
# Create a study that maximizes accuracy
study = optuna.create_study(direction='maximize')

# Run optimization for 50 trials
study.optimize(objective, n_trials=200, show_progress_bar=True)

# Best result
print("Best Score:", study.best_value)
print("Best Params:", study.best_params)

In [ ]:
# Train with best params
best_params = study.best_params
best_params

In [7]:
best_params = {'n_estimators': 800,
 'max_depth': 27,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'bootstrap': False}

In [8]:
# Train with best params
# best_params = study.best_params
rf_best = RandomForestClassifier(**best_params, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)

# Test accuracy
from sklearn.metrics import accuracy_score
y_pred = rf_best.predict(X_test)

In [9]:
print("Test Accuracy:", accuracy_score(y_test, y_pred))

Test Accuracy: 0.6311897106109324


In [ ]:
print(best_params)

| Parameter Space Size | Strategy | Approx Trials for Good Result |
| -------------------- | -------- | ----------------------------- |
| Small (3–4 params)   | Bayesian | 50–100 trials                 |
| Medium (5–7 params)  | Bayesian | 100–300 trials                |
| Large (8+ params)    | Bayesian | 300–1000 trials               |
| Random Search (any)  | —        | \~2–3× the Bayesian count     |


EVALUATIONS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
import seaborn as sns

In [ ]:
# Predictions
y_pred = rf_best.predict(X_test)
y_proba = rf_best.predict_proba(X_test)  # needed for curves

In [ ]:
# --- 1. Basic metrics ---
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
# --- 2. Confusion matrix ---
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=np.unique(y_test),
            yticklabels=np.unique(y_test))
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# --- 3. ROC Curve (One-vs-Rest) ---
classes = np.unique(y_test)
y_test_bin = label_binarize(y_test, classes=classes)
n_classes = y_test_bin.shape[1]

plt.figure(figsize=(7,6))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"Class {classes[i]} (AUC = {roc_auc:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curve (OvR)")
plt.legend(loc="lower right")
plt.show()

In [ ]:
# --- 4. Precision-Recall Curve (One-vs-Rest) ---
plt.figure(figsize=(7,6))
for i in range(n_classes):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_proba[:, i])
    ap_score = average_precision_score(y_test_bin[:, i], y_proba[:, i])
    plt.plot(recall, precision, lw=2, label=f"Class {classes[i]} (AP = {ap_score:.2f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Multiclass Precision-Recall Curve (OvR)")
plt.legend(loc="upper right")
plt.show()

What precision_recall_curve does
Given binary y_true and probability scores y_score:

It sorts predictions from highest probability to lowest.

At each threshold, it calculates:

Precision
=
TP
TP
+
FP
,
Recall
=
TP
TP
+
FN
Precision= 
TP+FP
TP
​
 ,Recall= 
TP+FN
TP
​
 
It returns arrays of recall and precision values for all thresholds.

In [ ]:
# --- 5. Macro & Weighted AUC ---
macro_auc = roc_auc_score(y_test_bin, y_proba, multi_class='ovr', average='macro')
weighted_auc = roc_auc_score(y_test_bin, y_proba, multi_class='ovr', average='weighted')
print(f"Macro-average AUC: {macro_auc:.4f}")
print(f"Weighted-average AUC: {weighted_auc:.4f}")

In [11]:
import onnxruntime as ort
print(ort.__version__)

1.22.1


In [12]:
import skl2onnx
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt
import numpy as np

# ----- 1. Convert model to ONNX (once) -----
initial_type = [("input", FloatTensorType([None, X_train.shape[1]]))]
onnx_model = convert_sklearn(rf_best, initial_types=initial_type)

with open("rf_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

In [13]:
# ----- 2. Load ONNX Runtime Session (CPU only, once) -----
sess = rt.InferenceSession(
    "rf_model.onnx",
    providers=["CPUExecutionProvider"]  # Ensures no GPU init overhead
)

input_name = sess.get_inputs()[0].name

In [20]:
# # ----- 3. Define reusable predict function -----
# def predict_onnx(sample):
#     """
#     Predict for a single sample or batch.
#     Input can be numpy array or pandas row.
#     """
#     arr = np.array(sample, dtype=np.float32).reshape(1, -1)
#     pred = sess.run(None, {input_name: arr})[0]
#     return pred

def predict_onnx(sample):
    """
    Predict for a single sample or batch.
    Input: Pandas DataFrame, Series, or NumPy array.
    """
    arr = np.array(sample, dtype=np.float32)
    
    # If single sample (1D), make it 2D
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
        
    return sess.run(None, {input_name: arr})[0]


In [18]:
# ----- 4. Example: Predict one sample -----
pred_class = predict_onnx(X_test.iloc[2])
print("Predicted class:", pred_class[0])

Predicted class: -1


In [21]:
# ----- 5. Example: Predict many samples without reloading -----
batch_preds = predict_onnx(X_test.iloc[0:10])
print("Batch predictions:", batch_preds)

Batch predictions: [-1  0 -1 -1  1  0  1  0 -1  1]


In [7]:
import joblib

# Save
joblib.dump(rf_best, "rf_model.joblib")

['rf_model.joblib']

In [8]:
import joblib

# Load the model once
loaded_model = joblib.load("rf_model.joblib")

# Select just the first row as a 2D array
one_sample = X_test.iloc[[0]]  # double brackets keep it 2D

# Predict
pred = loaded_model.predict(one_sample)

print("Predicted class:", pred[0])

Predicted class: -1


In [ ]:
loaded_model = joblib.load("rf_model.joblib")
preds = loaded_model.predict(X_test)

In [9]:
import pickle

# ----- Save -----
with open("rf_model.pkl", "wb") as f:
    pickle.dump(rf_best, f)

In [10]:
# ----- Load -----
with open("rf_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# Select just the first row as a 2D array
one_sample = X_test.iloc[[0]]  # keep 2D

# Predict
pred = loaded_model.predict(one_sample)

print("Predicted class:", pred[0])

Predicted class: -1
